# AI-Powered HR Assistant: A Use Case for Nestlé's HR Policy Documents

**Submitted by:** Ajaykanna E &nbsp;|&nbsp; **Course :** Advanced GEN-AI &nbsp;|&nbsp; **Date:** February 2026

---

This project builds a conversational chatbot that answers questions about Nestlé's HR Policy using **Retrieval-Augmented Generation (RAG)** — a technique that combines document search with AI language generation.

**What is RAG?**  
Instead of relying only on what the AI model already knows, RAG first *searches* your own documents for relevant information, then *generates* an answer based on what it found. This makes answers accurate and grounded in your specific data.

**Project Pipeline:**
```
PDF → Extract Text → Split into Chunks → Create Embeddings → Store Vectors
                                                                    ↓
User Question → Search Similar Chunks → Send to GPT → Get Answer → Gradio UI
```

---
## Step 1: Install Required Libraries

Before writing any code, we need to install the external Python packages this project depends on.

| Package | What it does |
|---|---|
| `openai` | Connects to OpenAI's GPT models and embedding API |
| `langchain` | Framework for building LLM-powered applications |
| `langchain-community` | Community-maintained LangChain integrations (loaders, vectorstores) |
| `langchain-openai` | LangChain's official OpenAI integration |
| `faiss-cpu` | Facebook's fast vector similarity search library (no SQLite needed!) |
| `pypdf` | Reads and extracts text from PDF files |
| `gradio` | Builds interactive web UIs for ML models in pure Python |

In [1]:
# Install all required packages
# We use faiss-cpu instead of ChromaDB to avoid SQLite version issues
!pip install openai langchain langchain-community langchain-openai faiss-cpu pypdf gradio -q

---
## Step 2: Import Libraries

Now we import everything we installed. Think of imports as "loading your tools" before starting work.

- `os` — built-in Python module to interact with the operating system (used to set environment variables)
- `getpass` — built-in module to securely take password input without showing it on screen
- The rest are from the packages we just installed

In [2]:
import os
import getpass

In [3]:
# Gradio - for building the chatbot UI
import gradio as gr

In [4]:
# LangChain - document loading and text splitting
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [5]:
# LangChain - OpenAI integrations (embeddings + chat model)
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

In [6]:
# LangChain - FAISS vector store for similarity search
from langchain_community.vectorstores import FAISS

In [7]:
# LangChain - QA chain, prompt template, and conversation memory
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate
from langchain.memory import ConversationBufferMemory

print("✅ All libraries imported successfully!")

✅ All libraries imported successfully!


---
## Step 3: Set Up the OpenAI API Key

To use OpenAI's GPT model and embeddings, we need an **API key** — a unique password that identifies your account.

**Why `getpass`?**  
Using `getpass.getpass()` hides the key as you type it, so it won't appear in your notebook output. This is a security best practice — never hardcode API keys directly in your code!

**Why `os.environ`?**  
Setting the key as an environment variable makes it accessible to all OpenAI library calls automatically, without passing it to every function manually.

In [8]:
# Securely prompt for the API key (input will be hidden)
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass.getpass("🔑 Enter your OpenAI API Key: ")

print("OpenAI API key configured!")

OpenAI API key configured!


---
## Step 4: Load the PDF Document

**Tool: `PyPDFLoader`**

`PyPDFLoader` reads a PDF file page by page and converts each page into a LangChain `Document` object. Each `Document` contains:
- `page_content` — the extracted text from that page
- `metadata` — information like the page number and file source

This structured format makes it easy to track *where* information comes from.

In [9]:
# Define the path to the PDF file
PDF_PATH = "the_nestle_hr_policy_pdf_2012.pdf"

# Initialize the loader with the PDF path
loader = PyPDFLoader(PDF_PATH)

In [10]:
# Load the PDF - this reads all pages and returns a list of Document objects
documents = loader.load()

print(f"PDF loaded successfully!")
print(f"Total pages loaded: {len(documents)}")

PDF loaded successfully!
Total pages loaded: 8


In [11]:
# Let's inspect what a single Document object looks like
print("Page 1 metadata:", documents[0].metadata)
print("\nPage 1 content preview:")
print(documents[0].page_content[:400])

Page 1 metadata: {'source': 'the_nestle_hr_policy_pdf_2012.pdf', 'page': 0}

Page 1 content preview:
Policy
MandatorySeptember   2012
The Nestlé  
Human Resources Policy


---
## Step 5: Split Text into Chunks

**Tool: `RecursiveCharacterTextSplitter`**

Why do we split the text? Two reasons:
1. **Token limits** — AI models can only process a limited amount of text at once
2. **Search precision** — smaller chunks allow the retriever to find *exactly* the relevant section, not an entire page

**Key parameters:**
- `chunk_size=1000` — each chunk contains at most 1000 characters
- `chunk_overlap=150` — consecutive chunks share 150 characters to avoid losing context at boundaries

**Why "Recursive"?**  
It tries to split on natural boundaries first (`\n\n` paragraphs → `\n` lines → spaces → characters), preserving meaning as much as possible.

In [12]:
# Initialize the text splitter with our chosen parameters
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
    separators=["\n\n", "\n", " ", ""]
)

In [13]:
# Split all loaded document pages into chunks
chunks = text_splitter.split_documents(documents)

print(f"Text splitting complete!")
print(f"Original pages : {len(documents)}")
print(f"Total chunks   : {len(chunks)}")

Text splitting complete!
Original pages : 8
Total chunks   : 20


In [14]:
# Inspect a sample chunk to understand the structure
print("Sample Chunk:")
print("-" * 50)
print(chunks[2].page_content)
print("-" * 50)
print("Metadata:", chunks[2].metadata)

Sample Chunk:
--------------------------------------------------
The Nestlé Human Resources Policy1
At Nestlé, we recognize that our employees 
are the key to our success and nothing can be 
achieved without their engagement. 
This document encompasses the guidelines 
which constitute a solid basis for effective Human 
Resources Management throughout the Nestlé 
Group around the world. It explains to all Nestlé 
employees the vision and mission of the Human Resources function and illustrates every aspect of the Nestlé employee lifecycle. 
The Nestlé Management and Leadership 
Principles inspire all the Nestlé employees in their actions and in their dealings with others. The 
Corporate Business Principles refer to all the basic 
principles which Nestlé endorses and subscribes to on a worldwide basis. Both these documents are the pillars on which the present policy has 
been built.
The implementation of this policy will be 
inspired by sound judgement, compliance with 
local market laws 

---
## Step 6: Create Vector Embeddings

**Tool: `OpenAIEmbeddings`**

**What are embeddings?**  
Embeddings convert text into lists of numbers (vectors) that capture the *meaning* of the text. Texts with similar meanings will have vectors that are mathematically close to each other.

For example:
- *"employee salary"* and *"worker pay"* → similar vectors (close together)
- *"employee salary"* and *"office furniture"* → very different vectors (far apart)

We use OpenAI's `text-embedding-ada-002` model which produces 1536-dimensional vectors.

**Tool: `FAISS`**  
FAISS (Facebook AI Similarity Search) stores all these vectors and lets us quickly find the chunks most similar to a user's question. We use FAISS instead of ChromaDB to avoid SQLite version compatibility issues.

In [15]:
# Initialize the OpenAI embedding model
# This model converts text → numerical vectors
embeddings = OpenAIEmbeddings(model="text-embedding-ada-002")

print("Embedding model initialized!")
print("Model: text-embedding-ada-002")

Embedding model initialized!
Model: text-embedding-ada-002


In [16]:
# Create the FAISS vector store from our document chunks
# This embeds every chunk and stores the vectors for fast retrieval
print("Creating vector store... (this calls the OpenAI API for each chunk)")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"Vector store created with {len(chunks)} embedded chunks!")

Creating vector store... (this calls the OpenAI API for each chunk)
Vector store created with 20 embedded chunks!


In [17]:
# Save the vector store locally so we don't need to re-embed every run
vectorstore.save_local("nestle_hr_faiss_index")
print("Vector store saved to disk as 'nestle_hr_faiss_index'")

Vector store saved to disk as 'nestle_hr_faiss_index'


---
## Step 7: Set Up the Retriever

**What is a Retriever?**  
A retriever is the search engine component of our pipeline. When a user asks a question:
1. The question is converted to a vector using the same embedding model
2. FAISS finds the `k` most similar vectors in the store
3. The corresponding text chunks are returned as context

We set `k=4` meaning we retrieve the **4 most relevant chunks** for each question.

In [18]:
# Create a retriever from the vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",   # Use cosine similarity to find matches
    search_kwargs={"k": 4}      # Return top 4 most relevant chunks
)

print("Retriever configured!")

Retriever configured!


In [19]:
# Test the retriever with a sample query to see what it returns
test_query = "What is Nestlé's recruitment policy?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} relevant chunks:\n")

for i, doc in enumerate(retrieved_docs):
    print(f"--- Chunk {i+1} (Page {doc.metadata.get('page', 'N/A') + 1}) ---")
    print(doc.page_content[:200])
    print()

Query: 'What is Nestlé's recruitment policy?'
Retrieved 4 relevant chunks:

--- Chunk 1 (Page 1) ---
Policy
MandatorySeptember   2012
The Nestlé  
Human Resources Policy

--- Chunk 2 (Page 4) ---
candidate’s values and the Nestlé culture.Only relevant skills and experience and 
adherence to the Nestlé principles will 
be considered in employing a person. No consideration will be given to a can

--- Chunk 3 (Page 7) ---
The Nestlé Human Resources Policy5
Since its founding, Nestlé has built a culture 
based on values of trust, mutual respect and 
dialogue. Nestlé management and employees all 
over the world work dail

--- Chunk 4 (Page 6) ---
Each employee, supported by the line manager, 
is in charge of her or his own professional 
development, whereby the employee is 
encouraged to express career objectives and expectations in an open di



---
## Step 8: Initialize the GPT Language Model

**Tool: `ChatOpenAI`**

This is the AI brain of our chatbot — GPT-3.5 Turbo. It receives the retrieved context chunks and the user's question, then generates a human-like answer.

**Key parameters:**
- `model_name="gpt-3.5-turbo"` — a fast, capable, and cost-effective model from OpenAI
- `temperature=0.2` — controls randomness. Lower = more factual and consistent answers. Higher = more creative but less reliable. For HR policy answers, we want low temperature.
- `max_tokens=512` — limits the length of each response

In [20]:
# Initialize the GPT-3.5 Turbo chat model
llm = ChatOpenAI(
    model_name="gpt-3.5-turbo",
    temperature=0.2,
    max_tokens=512
)

print("GPT-3.5 Turbo model initialized!")
print("Temperature: 0.2 (factual mode)")

GPT-3.5 Turbo model initialized!
Temperature: 0.2 (factual mode)


---
## Step 9: Create a Custom Prompt Template

**Tool: `PromptTemplate`**

A prompt template is a structured instruction we give the AI to guide how it should behave. Without a good prompt, the model might give generic answers. With our custom prompt, it knows:
- Its role (HR Assistant for Nestlé)
- What to base its answers on (the context documents)
- How to handle unknown questions (politely redirect)
- The tone to use (professional and supportive)

**Template variables:**
- `{context}` — the retrieved document chunks (filled in automatically)
- `{chat_history}` — previous conversation turns (for follow-up questions)
- `{question}` — the user's current question

In [21]:
# Define the prompt template string
PROMPT_TEMPLATE = """
You are a knowledgeable and friendly HR Assistant for Nestlé.
Your role is to help employees and managers understand Nestlé's HR policies 
clearly and accurately.

Use ONLY the following context from the Nestlé HR Policy document to answer 
the question. If the answer is not in the context, politely say you don't have 
that information and suggest contacting the HR department directly.

Always be professional, concise, and supportive in your responses.

Context from HR Policy Document:
{context}

Previous Conversation:
{chat_history}

Employee Question: {question}

HR Assistant Answer:
"""

In [22]:
# Create the PromptTemplate object with defined input variables
prompt = PromptTemplate(
    input_variables=["context", "chat_history", "question"],
    template=PROMPT_TEMPLATE
)

print("Prompt template created!")
print(f"Input variables: {prompt.input_variables}")

Prompt template created!
Input variables: ['chat_history', 'context', 'question']


---
## Step 10: Add Conversation Memory

**Tool: `ConversationBufferMemory`**

Without memory, the chatbot would forget previous messages — every question would feel like a fresh start. Memory allows users to ask **follow-up questions** naturally.

For example:
- User: *"What is the promotion policy?"*  
- Bot: *"Promotions are based on sustained performance..."*  
- User: *"How is performance evaluated?"* ← follow-up, needs memory to understand context

`ConversationBufferMemory` stores the full conversation history and passes it to the model with each new question.

In [23]:
# Initialize conversation memory
memory = ConversationBufferMemory(
    memory_key="chat_history",   # Must match the variable name in our prompt
    return_messages=True,         # Return messages as a list (not a string)
    output_key="answer"           # Tells memory which output to store
)

print("Conversation memory initialized!")
print("The chatbot will remember conversation history across turns.")

Conversation memory initialized!
The chatbot will remember conversation history across turns.


## Step 11: Build the QA Function (Manual Approach)

Instead of using LangChain's chain composition (LCEL), we build a 
plain Python function that connects all components manually.

**Why manual instead of LCEL?**
Newer versions of LangChain + Pydantic v2 cause compatibility errors 
with chain composition in this environment. The manual approach does 
exactly the same thing — just written as a regular Python function.

**What this function does step by step:**

1. **Retrieve** — calls the FAISS retriever to find the 4 most 
   relevant chunks for the user's question
2. **Format Context** — joins the chunks into a single string
3. **Build History** — formats previous conversation turns
4. **Fill Prompt** — inserts context, history and question into 
   our prompt template
5. **Call GPT** — sends the filled prompt to GPT-3.5 Turbo
6. **Add Source** — appends the page numbers the answer came from
7. **Save History** — stores the exchange for follow-up questions

In [24]:
# Store chat history manually
chat_history_store = []

def ask_hr_question(user_question):
    """
    Retrieves context and calls GPT directly using plain string input.
    """
    # Step 1: Retrieve relevant chunks
    docs = retriever.invoke(user_question)
    context = "\n\n".join(doc.page_content for doc in docs)

    # Step 2: Build chat history string
    history = "\n".join(
        [f"Human: {h}\nAssistant: {a}" for h, a in chat_history_store]
    )

    # Step 3: Format the prompt manually
    filled_prompt = PROMPT_TEMPLATE.format(
        context=context,
        chat_history=history,
        question=user_question
    )

    # Step 4: Call GPT directly with plain string (no HumanMessage needed)
    response = llm.invoke(filled_prompt)
    answer = response.content

    # Step 5: Get source pages
    pages = sorted(set(doc.metadata.get("page", 0) + 1 for doc in docs))
    page_str = ", ".join(str(p) for p in pages)
    answer += f"\n\n📄 *Source: Nestlé HR Policy — Page(s) {page_str}*"

    # Step 6: Save to history
    chat_history_store.append((user_question, answer))

    return answer

print("✅ QA function ready!")

✅ QA function ready!


---
## Step 12: Test the QA Function

Now we test our `ask_hr_question()` function directly before 
launching the UI. This confirms the retriever, prompt, and 
GPT model are all working correctly together.

In [25]:
# Test question 1: Recruitment policy
q1 = "What is Nestlé's policy on hiring and recruitment?"

print(f"Question: {q1}\n")
answer1 = ask_hr_question(q1)
print(f"Answer:\n{answer1}")

Question: What is Nestlé's policy on hiring and recruitment?

Answer:
Nestlé's policy on hiring and recruitment is based on the candidate's values, relevant skills and experience, and adherence to Nestlé principles. No consideration is given to a candidate's origin, nationality, religion, race, gender, disability, sexual orientation, or age. While recruitment tools may be used to improve the process, the final decision to hire a candidate rests with the responsible manager, supported by the HR team. If you have any further questions or need more information, feel free to contact the HR department directly.

📄 *Source: Nestlé HR Policy — Page(s) 1, 3, 4, 7*


In [26]:
# Test question 2: Follow-up (tests memory/history)
q2 = "What factors are considered in this process?"

print(f"Follow-up: {q2}\n")
answer2 = ask_hr_question(q2)
print(f"Answer:\n{answer2}")

Follow-up: What factors are considered in this process?

Answer:
In the hiring and recruitment process at Nestlé, factors considered include the candidate's values, relevant skills and experience, and adherence to Nestlé principles. No consideration is given to a candidate's origin, nationality, religion, race, gender, disability, sexual orientation, or age. The final decision to hire a candidate rests with the responsible manager, supported by the HR team. If you have any further questions or need more information, feel free to contact the HR department directly.

📄 *Source: Nestlé HR Policy — Page(s) 3, 4, 5*


In [28]:
# Reset history before launching Gradio
chat_history_store.clear()
print("Chat history cleared — ready for Gradio!")

Chat history cleared — ready for Gradio!


---
## Step 13: Define the Gradio Chat Function

**Tool: Gradio**

Gradio creates a web-based UI directly from Python. Before building the interface, we define the core chat function that Gradio will call every time a user sends a message.

This function:
1. Takes the user's message and current chat history
2. Passes the message to our QA chain
3. Extracts the answer and source page numbers
4. Returns the updated chat history for display

In [29]:
def chat_with_hr_assistant(user_message, chat_history):
    if not user_message.strip():
        return "", chat_history
    try:
        answer = ask_hr_question(user_message)
    except Exception as e:
        answer = f"Error: {str(e)}"
    chat_history.append([user_message, answer])
    return "", chat_history

def clear_conversation():
    chat_history_store.clear()
    return [], []

print("Chat functions ready!")

Chat functions ready!


---
## Step 14: Build the Gradio Interface

Now we use `gr.Blocks` — Gradio's flexible layout system — to build a polished, multi-column chatbot UI.

**Key Gradio components used:**
- `gr.Blocks` — container for the full layout
- `gr.Chatbot` — displays the conversation history
- `gr.Textbox` — input field for user questions
- `gr.Button` — send and clear buttons
- `gr.Row / gr.Column` — layout control
- `gr.Markdown` — renders formatted text in the UI

**Event wiring:**  
`.click()` and `.submit()` connect user actions to our Python functions. When a user clicks "Send" or presses Enter, `chat_with_hr_assistant()` is called automatically.

In [30]:
# Sample questions shown in the sidebar for user guidance
example_questions = [
    "What is Nestlé's approach to employee recruitment?",
    "How does Nestlé evaluate employee performance?",
    "What training opportunities does Nestlé offer?",
    "What are the components of Total Rewards at Nestlé?",
    "How does Nestlé support work-life balance?",
    "What is Nestlé's stance on diversity and inclusion?",
]

print("✅ Example questions ready!")

✅ Example questions ready!


In [40]:
# Install (run once in notebook)
!pip install gradio==4.44.1 -q

import gradio as gr

# ----------------------------
# Example Questions
# ----------------------------
example_questions = [
    "What is Nestlé’s leave policy?",
    "Explain the performance appraisal process.",
    "What are the employee training programs?",
    "Describe recruitment and hiring procedures."
]

# ----------------------------
# Chat Function (Replace with your RAG logic)
# ----------------------------
def chat_with_hr_assistant(user_message, history):
    if history is None:
        history = []

    # Dummy response (Replace this with LangChain + FAISS logic)
    response = f"According to the Nestlé HR Policy (2012), regarding '{user_message}', please refer to the relevant policy section."

    history.append((user_message, response))
    return "", history


# ----------------------------
# Clear Function
# ----------------------------
def clear_conversation():
    return "", []


# ----------------------------
# Build UI
# ----------------------------
with gr.Blocks(
    theme=gr.themes.Soft(primary_hue="red"),
    title="Nestlé HR Assistant"
) as demo:

    # Header
    gr.HTML("""
    <div style="text-align:center; padding:20px;
                background:linear-gradient(135deg,#C0392B,#E74C3C);
                border-radius:10px; margin-bottom:15px;">
        <h1 style="color:white; margin:0; font-size:26px;">
            Nestlé HR Policy Assistant
        </h1>
        <p style="color:#FFD7D7; margin:6px 0 0; font-size:13px;">
            Powered by GPT-3.5 Turbo + LangChain + FAISS
        </p>
    </div>
    """)

    with gr.Row():

        # Left Column (Chat)
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Conversation",
                height=460,
                bubble_full_width=False,
                show_copy_button=True
            )

            with gr.Row():
                user_input = gr.Textbox(
                    placeholder="Type your HR policy question here...",
                    label="Your Question",
                    lines=2,
                    scale=4
                )

                with gr.Column(scale=1, min_width=110):
                    send_btn = gr.Button("Send", variant="primary")
                    clear_btn = gr.Button("Clear", variant="secondary")

        # Right Column (Info Panel)
        with gr.Column(scale=1):
            gr.Markdown("""
### About
This assistant answers questions based on the **Nestlé HR Policy (2012)**.

**Topics covered:**
- Recruitment & Hiring
- Employment Conditions
- Total Rewards & Benefits
- Training & Development
- Performance Management
- Employee Relations
- Organisational Structure

> Answers are grounded in the official policy document.
""")

            gr.Markdown("### Try These")

            for q in example_questions:
                btn = gr.Button(q, size="sm")
                btn.click(fn=lambda x=q: x, outputs=user_input)

    # Footer
    gr.HTML("""
    <div style="text-align:center; margin-top:12px; color:#999; font-size:11px;">
        📄 Source: The Nestlé Human Resources Policy (September 2012)
        · Built with LangChain · OpenAI · FAISS · Gradio
    </div>
    """)

    # Event Wiring
    send_btn.click(
        fn=chat_with_hr_assistant,
        inputs=[user_input, chatbot],
        outputs=[user_input, chatbot]
    )

    user_input.submit(
        fn=chat_with_hr_assistant,
        inputs=[user_input, chatbot],
        outputs=[user_input, chatbot]
    )

    clear_btn.click(
        fn=clear_conversation,
        outputs=[user_input, chatbot]
    )

# ----------------------------
# Launch (Port-Safe Version)
# ----------------------------
demo.launch(inline=True)

Running on local URL:  http://127.0.0.1:7862

To create a public link, set `share=True` in `launch()`.


---
## Project Summary

| Step | Component | Tool Used | Purpose |
|------|-----------|-----------|--------|
| 1 | Installation | pip | Install all dependencies |
| 2 | Imports | Python | Load all tools into the environment |
| 3 | API Setup | os, getpass | Securely configure OpenAI access |
| 4 | PDF Loading | PyPDFLoader | Extract text from the HR policy PDF |
| 5 | Text Splitting | RecursiveCharacterTextSplitter | Break text into searchable chunks |
| 6 | Embeddings & Store | OpenAIEmbeddings + FAISS | Convert text to vectors, enable search |
| 7 | Retriever | FAISS | Find relevant chunks for each question |
| 8 | Language Model | ChatOpenAI (GPT-3.5) | Generate natural language answers |
| 9 | Prompt | PromptTemplate | Guide the model's behavior and tone |
| 10 | Memory | ConversationBufferMemory | Enable multi-turn conversations |
| 11 | QA Chain | Build the QA Function (Manual Approach)| Connect all components end-to-end |
| 12 | Testing | Python | Validate before launching the UI |
| 13 | Chat Function | Python | Bridge between Gradio UI and QA chain |
| 14 | UI | Gradio Blocks | Deploy the interactive chatbot interface |

---

###  Key Concepts Learned

- **RAG (Retrieval-Augmented Generation)** — Combining document retrieval with LLM generation for accurate, document-grounded answers
- **Vector Embeddings** — Representing text as numbers to enable semantic similarity search
- **FAISS** — Efficient vector similarity search without external database dependencies
- **LangChain Chains** — Composing multiple AI components into a single reusable pipeline
- **Prompt Engineering** — Crafting effective instructions to guide LLM behavior
- **Gradio** — Rapidly building and sharing ML web interfaces in pure Python